# Recommendation Quality Evaluation

> **Author:** Umesh Pandey  
> **Project:** Semantic Movie Recommendation (B.Tech NLP Portfolio Project)

This notebook implements and evaluates standard information-retrieval metrics:

| Metric | What it measures |
|---|---|
| **Precision@K** | Fraction of top-K results that are relevant |
| **Recall@K** | Fraction of all relevant items found in top-K |
| **NDCG@K** | Ranking quality, rewarding relevant items appearing earlier |

We evaluate at `K = 5` and `K = 10` over 10 curated test queries.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from typing import List
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')

print('Setup complete.')

## 1. Metric Implementations

We implement the three standard metrics from scratch, mirroring `backend/evaluation.py`.

In [ ]:
def precision_at_k(recommended_ids: List[int], relevant_ids: List[int], k: int) -> float:
    """
    Precision@K: fraction of the top-K recommended items that are relevant.
    
    P@K = |{recommended_ids[:k]} ∩ {relevant_ids}| / K
    """
    if k <= 0:
        raise ValueError('k must be a positive integer.')
    if not recommended_ids:
        return 0.0
    top_k = set(recommended_ids[:k])
    hits = top_k & set(relevant_ids)
    return len(hits) / k


def recall_at_k(recommended_ids: List[int], relevant_ids: List[int], k: int) -> float:
    """
    Recall@K: fraction of all relevant items appearing in the top-K results.
    
    R@K = |{recommended_ids[:k]} ∩ {relevant_ids}| / |relevant_ids|
    """
    if not relevant_ids or not recommended_ids:
        return 0.0
    top_k = set(recommended_ids[:k])
    hits = top_k & set(relevant_ids)
    return len(hits) / len(relevant_ids)


def ndcg_at_k(recommended_ids: List[int], relevant_ids: List[int], k: int) -> float:
    """
    NDCG@K: Normalised Discounted Cumulative Gain.
    Uses binary relevance (1 if relevant, 0 if not).
    
    DCG@K  = Σ [rel_i / log2(i+1)]
    IDCG@K = Σ [1 / log2(i+1)]  for min(K, |relevant|) perfect hits
    NDCG@K = DCG@K / IDCG@K
    """
    if not recommended_ids or not relevant_ids:
        return 0.0
    relevant_set = set(relevant_ids)
    dcg = sum(
        1.0 / math.log2(rank + 1)
        for rank, mid in enumerate(recommended_ids[:k], start=1)
        if mid in relevant_set
    )
    ideal_hits = min(len(relevant_ids), k)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


# Quick sanity check
recs = [1, 4, 2, 8, 15, 3, 6, 11, 20, 5]
rels = [1, 4, 8, 11, 25, 30]
print('=== Sanity Check (K=10) ===')
print(f'Recommended : {recs}')
print(f'Relevant    : {rels}')
print(f'P@10        : {precision_at_k(recs, rels, 10):.4f}  (expected: 4/10 = 0.4000)')
print(f'R@10        : {recall_at_k(recs, rels, 10):.4f}  (expected: 4/6 = 0.6667)')
print(f'NDCG@10     : {ndcg_at_k(recs, rels, 10):.4f}')

## 2. Test Queries and Ground Truth

Each test query comes with a manually curated set of `relevant_ids` — the `movie_id` values from our metadata CSV that are semantically relevant to that query. These serve as our ground truth for evaluation.

In [ ]:
TEST_QUERIES = [
    {
        'query': 'space exploration science fiction',
        'relevant_ids': [4, 8, 15, 101, 110, 125, 140],
        'description': 'Sci-fi and space films',
    },
    {
        'query': 'prison escape redemption friendship',
        'relevant_ids': [1, 5, 7, 50, 65],
        'description': 'Drama about friendship and resilience',
    },
    {
        'query': 'superhero crime fighting dark city',
        'relevant_ids': [2, 22, 30, 45, 60],
        'description': 'Action/thriller superhero films',
    },
    {
        'query': 'animated fantasy magical journey children',
        'relevant_ids': [10, 80, 90, 120, 130],
        'description': 'Animated fantasy adventures',
    },
    {
        'query': 'psychological thriller mind bending twist',
        'relevant_ids': [3, 11, 12, 40, 55],
        'description': 'Mind-bending thrillers',
    },
    {
        'query': 'romantic love story music dancing',
        'relevant_ids': [13, 70, 75, 85, 95],
        'description': 'Romance and musical films',
    },
    {
        'query': 'sports underdog triumph competition',
        'relevant_ids': [17, 19, 35, 48, 62],
        'description': 'Sports drama and underdog stories',
    },
    {
        'query': 'Indian comedy social commentary satire',
        'relevant_ids': [18, 20, 100, 112],
        'description': 'Indian comedy-drama films',
    },
    {
        'query': 'post apocalyptic survival action wasteland',
        'relevant_ids': [14, 25, 38, 56, 78],
        'description': 'Post-apocalyptic action films',
    },
    {
        'query': 'true story biography genius mathematics',
        'relevant_ids': [9, 16, 43, 67, 88],
        'description': 'Biographical dramas about real-life brilliance',
    },
]

print(f'Number of test queries: {len(TEST_QUERIES)}')
for q in TEST_QUERIES:
    print(f"  [{q['description'][:45]}]  relevant_ids={q['relevant_ids']}")

In [ ]:
import hashlib

def make_fake_embedding(text: str, dim: int = 384) -> np.ndarray:
    """Deterministic hash-based fake embedding (no GPU needed)."""
    digest = hashlib.sha256(text.encode()).digest()
    seed = int.from_bytes(digest, 'big') % (2**32)
    rng = np.random.default_rng(seed)
    vec = rng.standard_normal(dim).astype(np.float32)
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec


# Load metadata and build fake embedding corpus
METADATA_PATH = Path('../embeddings/movies_metadata.csv')
df = pd.read_csv(METADATA_PATH)

# Build corpus embeddings
docs = [f"{row['title']} {row['genre']} {row['plot']}" for _, row in df.iterrows()]
corpus_embeddings = np.array([make_fake_embedding(doc) for doc in docs])
movie_ids = df['movie_id'].tolist()

print(f'Corpus shape: {corpus_embeddings.shape}')

## 3. Evaluate at K = 5 and K = 10

In [ ]:
def evaluate_at_k(test_queries, corpus_embeddings, movie_ids, k):
    """Run evaluation over all test queries at a given K."""
    results = []
    for entry in test_queries:
        query_vec = make_fake_embedding(entry['query'])
        scores = corpus_embeddings @ query_vec  # cosine similarity (unit vecs)
        
        # Top-K indices by descending score
        top_k_idx = np.argsort(scores)[::-1][:k]
        recommended_ids = [movie_ids[i] for i in top_k_idx]
        
        p_k = precision_at_k(recommended_ids, entry['relevant_ids'], k)
        r_k = recall_at_k(recommended_ids, entry['relevant_ids'], k)
        n_k = ndcg_at_k(recommended_ids, entry['relevant_ids'], k)
        
        results.append({
            'query': entry['query'],
            'description': entry['description'],
            f'P@{k}': p_k,
            f'R@{k}': r_k,
            f'NDCG@{k}': n_k,
        })
    return pd.DataFrame(results)


results_k5  = evaluate_at_k(TEST_QUERIES, corpus_embeddings, movie_ids, k=5)
results_k10 = evaluate_at_k(TEST_QUERIES, corpus_embeddings, movie_ids, k=10)

# Display K=5
print('=== Evaluation at K=5 ===')
display_cols = ['description', 'P@5', 'R@5', 'NDCG@5']
print(results_k5[display_cols].to_string(index=False))
print(f"\nMean P@5  : {results_k5['P@5'].mean():.4f}")
print(f"Mean R@5  : {results_k5['R@5'].mean():.4f}")
print(f"Mean NDCG@5: {results_k5['NDCG@5'].mean():.4f}")

In [ ]:
print('=== Evaluation at K=10 ===')
display_cols = ['description', 'P@10', 'R@10', 'NDCG@10']
print(results_k10[display_cols].to_string(index=False))
print(f"\nMean P@10  : {results_k10['P@10'].mean():.4f}")
print(f"Mean R@10  : {results_k10['R@10'].mean():.4f}")
print(f"Mean NDCG@10: {results_k10['NDCG@10'].mean():.4f}")

## 4. Visualise Metric Comparisons

In [ ]:
# Aggregate means for bar chart
metrics = {
    'P@5' : results_k5['P@5'].mean(),
    'P@10': results_k10['P@10'].mean(),
    'R@5' : results_k5['R@5'].mean(),
    'R@10': results_k10['R@10'].mean(),
    'NDCG@5': results_k5['NDCG@5'].mean(),
    'NDCG@10': results_k10['NDCG@10'].mean(),
}

labels = list(metrics.keys())
values = list(metrics.values())

colors = ['#4e79a7', '#f28e2b', '#59a14f', '#e15759', '#76b7b2', '#edc948']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: all 6 metric bars ---
axes[0].bar(labels, values, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_ylim(0, 1.0)
axes[0].set_title('Mean Metrics — All 10 Test Queries', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Score')
axes[0].set_xlabel('Metric')
axes[0].grid(axis='y', alpha=0.4)
for bar, val in zip(axes[0].patches, values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f'{val:.3f}',
        ha='center', fontsize=9,
    )

# --- Right: per-query NDCG@10 ---
descs = [q['description'][:30] for q in TEST_QUERIES]
ndcg_vals = results_k10['NDCG@10'].values
bar_colors = plt.cm.RdYlGn(ndcg_vals / ndcg_vals.max())

axes[1].barh(descs[::-1], ndcg_vals[::-1], color=bar_colors[::-1], edgecolor='white')
axes[1].set_xlim(0, 1.0)
axes[1].set_title('Per-Query NDCG@10', fontweight='bold', fontsize=12)
axes[1].set_xlabel('NDCG@10')
axes[1].grid(axis='x', alpha=0.4)
for val, y_pos in zip(ndcg_vals[::-1], range(len(descs))):
    axes[1].text(val + 0.01, y_pos, f'{val:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('evaluation_metrics.png', bbox_inches='tight')
plt.show()
print('Evaluation figure saved.')

In [ ]:
# Summary table
summary = pd.DataFrame({
    'Metric': ['Precision@K', 'Recall@K', 'NDCG@K'],
    'K=5' : [
        f"{results_k5['P@5'].mean():.4f}",
        f"{results_k5['R@5'].mean():.4f}",
        f"{results_k5['NDCG@5'].mean():.4f}",
    ],
    'K=10': [
        f"{results_k10['P@10'].mean():.4f}",
        f"{results_k10['R@10'].mean():.4f}",
        f"{results_k10['NDCG@10'].mean():.4f}",
    ],
})
print('=== Summary Table ===')
print(summary.to_string(index=False))

## Analysis

### Interpretation of Results

| Observation | Explanation |
|---|---|
| **P@10 < P@5** | Precision naturally decreases as K increases — more room for irrelevant results |
| **R@10 > R@5** | Recall increases with K — retrieving more items covers more ground truth |
| **NDCG@10 ≥ NDCG@5** | NDCG rewards relevant items appearing early; K=10 gives more opportunity |

### Metric Trade-offs

- **Precision@K** is ideal when users inspect only the top-K results and we want minimal irrelevant noise.
- **Recall@K** is ideal when it's critical not to miss any relevant items (e.g., medical literature retrieval).
- **NDCG@K** balances both by accounting for rank position — the earlier a relevant result appears, the better.

### Limitations of this Evaluation

1. **Synthetic embeddings:** This notebook uses hash-based fake embeddings. Numbers will differ significantly with real `sentence-transformers` embeddings.
2. **Manual ground truth:** The `relevant_ids` were manually curated and are inherently subjective.
3. **Small test set:** 10 queries is sufficient for a demo but insufficient for statistical significance in production.

### Recommended Next Steps

- Run `python backend/generate_embeddings.py` and reload this notebook with real embeddings.
- Expand the test query set to 50+ queries with crowdsourced relevance judgments.
- Experiment with different embedding models and compare their NDCG@10 scores.
- Consider incorporating rating as a re-ranking signal (score = α·cosine_sim + β·normalised_rating).